fetch live weather data forecast for boston right now

In [1]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path
from datetime import datetime, timedelta

sys.path.insert(0, str(Path.cwd().parent))

from feature_store.sources.weather import fetch_weather_forecast, fetch_historical_weather
from feature_store.sources.events import generate_boston_events, events_to_features

# Get the REAL 7-day forecast for Boston RIGHT NOW
forecast = fetch_weather_forecast()

print("🌤️  LIVE Boston Weather Forecast")
print("=" * 60)
for _, row in forecast.iterrows():
    dt = pd.Timestamp(row["date"])
    rain = "🌧️" if row["is_precipitation"] else "☀️"
    print(f"  {dt.strftime('%A %b %d'):>20s}  |  {row['temp_high']:.0f}°F  |  "
          f"Feels like {row['temp_feels_like']:.0f}°F  |  {rain} "
          f"{'Rain' if row['is_precipitation'] else 'Dry'} "
          f"({row.get('rain_probability', 0):.0f}%)")

🌤️  LIVE Boston Weather Forecast
        Tuesday Mar 03  |  36°F  |  Feels like 27°F  |  🌧️ Rain (100%)
      Wednesday Mar 04  |  51°F  |  Feels like 47°F  |  🌧️ Rain (83%)
       Thursday Mar 05  |  36°F  |  Feels like 29°F  |  🌧️ Rain (61%)
         Friday Mar 06  |  35°F  |  Feels like 28°F  |  🌧️ Rain (81%)
       Saturday Mar 07  |  48°F  |  Feels like 42°F  |  🌧️ Rain (43%)
         Sunday Mar 08  |  50°F  |  Feels like 44°F  |  🌧️ Rain (43%)
         Monday Mar 09  |  60°F  |  Feels like 53°F  |  ☀️ Dry (12%)


get upcomoing boston events

In [2]:
# Generate realistic Boston events for the next 7 days
# (In production, this would be Ticketmaster API)
today = datetime.now().strftime("%Y-%m-%d")
next_week = (datetime.now() + timedelta(days=7)).strftime("%Y-%m-%d")

events = generate_boston_events(today, next_week, seed=int(datetime.now().timestamp()) % 1000)

print("🎫  Upcoming Boston Events (Next 7 Days)")
print("=" * 70)
if events.empty:
    print("  No events found")
else:
    for _, row in events.iterrows():
        dt = pd.Timestamp(row["date"])
        print(f"  {dt.strftime('%A %b %d'):>20s}  |  {row['name']:<35s}  |  "
              f"~{row['attendance_est']:,} people  |  {row['distance_miles']:.1f} mi")

# Convert to daily features
forecast_dates = forecast["date"].values
event_features = events_to_features(events, forecast_dates)
print(f"\nEvent features for forecast period:")
print(event_features.to_string(index=False))

🎫  Upcoming Boston Events (Next 7 Days)
      Wednesday Mar 04  |  Celtics Home Game                    |  ~19,799 people  |  4.5 mi
      Wednesday Mar 04  |  Bruins Home Game                     |  ~17,363 people  |  4.5 mi
       Thursday Mar 05  |  Concert at House of Blues            |  ~1,857 people  |  4.0 mi
       Saturday Mar 07  |  College Sports Event                 |  ~5,784 people  |  4.8 mi
        Tuesday Mar 10  |  Concert at Leader Bank Pavilion      |  ~3,252 people  |  3.5 mi

Event features for forecast period:
      date  total_events  total_attendance  max_attendance  has_sports  has_music  nearby_event
2026-03-03             0                 0               0           0          0             0
2026-03-04             2             37162           19799           1          0             0
2026-03-05             1              1857            1857           0          1             0
2026-03-06             0                 0               0           0       

Load the Boston store data and train the model:

In [3]:
import xgboost as xgb
from feature_store.engineer import build_features, get_feature_columns
from model.evaluate import compute_wmape

BOSTON_DATA = Path(r"C:\Users\syeds\OneDrive\Desktop\nboracle\nb_oracle\data\raw\boston-bodega")

train_df = pd.read_csv(BOSTON_DATA / "train.csv", parse_dates=["date"])
holidays = pd.read_csv(BOSTON_DATA / "holidays_events.csv", parse_dates=["date"])
weather_hist = pd.read_csv(BOSTON_DATA / "weather_pipeline.csv", parse_dates=["date"])

# Train on beverages
store1 = train_df[train_df.store_nbr == 1].copy()
bev = store1[store1.family == "BEVERAGES"].sort_values("date").reset_index(drop=True)

features = build_features(bev, holidays, weather_df=weather_hist).dropna()
feat_cols = get_feature_columns(features)

# Use last 30 days as test
split = (pd.Timestamp(today) - pd.Timedelta(days=30)).strftime("%Y-%m-%d")
train_data = features[features.index < split]
test_data = features[features.index >= split]

X_train, y_train = train_data[feat_cols], train_data["sales"]
X_test, y_test = test_data[feat_cols], test_data["sales"]

model = xgb.XGBRegressor(
    objective="reg:tweedie", tweedie_variance_power=1.6,
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=10,
    reg_alpha=0.1, reg_lambda=1.0, random_state=42,
    early_stopping_rounds=30)

model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print(f"✅ Model trained on {len(X_train)} days of Boston data")

wmape = compute_wmape(y_test.values, np.maximum(model.predict(X_test), 0))
print(f"   Test WMAPE: {wmape:.1%}")

✅ Model trained on 734 days of Boston data
   Test WMAPE: 34.4%


Make LIVE predictions for the next 7 days:

In [5]:
from explainability.translator import get_shap_explanation, format_whatsapp_message

recent_sales = bev.tail(60).copy()

print("🔮  LIVE 7-DAY FORECAST — Dorchester Bodega — Beverages")
print("=" * 65)

for _, fday in forecast.iterrows():
    forecast_date = pd.Timestamp(fday["date"])
    
    # Create future date row
    future_row = pd.DataFrame({
        "date": [forecast_date],
        "sales": [0],
        "onpromotion": [0],
    })
    
    # Combine recent history + future date
    combined = pd.concat([
        recent_sales[["date", "sales", "onpromotion"]],
        future_row
    ]).reset_index(drop=True)
    
    # Build weather ONLY for the dates in combined
    combined_dates = pd.to_datetime(combined["date"])
    weather_for_combined = weather_hist[weather_hist["date"].isin(combined_dates)].copy()
    
    # Add forecast day's weather
    weather_row = pd.DataFrame({
        "date": [forecast_date],
        "temp_high": [fday["temp_high"]],
        "temp_feels_like": [fday["temp_feels_like"]],
        "precipitation_mm": [fday["precipitation_mm"]],
        "is_precipitation": [fday["is_precipitation"]],
        "humidity": [65],
    })
    
    weather_for_combined = pd.concat([weather_for_combined, weather_row]).drop_duplicates(subset="date")
    
    try:
        future_features = build_features(combined, holidays, weather_df=weather_for_combined).dropna()
        
        if len(future_features) == 0:
            print(f"  {forecast_date.strftime('%A %b %d')}: Could not generate features")
            continue
        
        last_row = future_features.iloc[-1]
        feature_row = last_row[feat_cols]
        
        prediction = max(model.predict(feature_row.values.reshape(1, -1))[0], 0)
        
        result = get_shap_explanation(model, feature_row, feat_cols, forecast_date, prediction)
        
        # Event info
        day_events = events[events.date == forecast_date.strftime("%Y-%m-%d")] if not events.empty else pd.DataFrame()
        event_str = ""
        if not day_events.empty:
            event_str = f" | 🎫 {', '.join(day_events['name'].values)}"
        
        rain = "🌧️" if fday["is_precipitation"] else "☀️"
        conf_emoji = {"high": "🟢", "moderate": "🟡", "low": "🔴"}
        
        print(f"\n  📅 {forecast_date.strftime('%A, %B %d')}")
        print(f"     🌡️ {fday['temp_high']:.0f}°F  {rain}{event_str}")
        print(f"     📦 Predicted: ~{prediction:.0f} units")
        print(f"     {conf_emoji.get(result['confidence'], '⚪')} {result['summary']}")
        print(f"     💬 {result['explanation']}")
        
    except Exception as e:
        print(f"  {forecast_date.strftime('%A %b %d')}: Error — {e}")

🔮  LIVE 7-DAY FORECAST — Dorchester Bodega — Beverages

  📅 Tuesday, March 03
     🌡️ 36°F  🌧️
     📦 Predicted: ~97 units
     🟢 Expecting a quieter Tuesday — forecast is ~97 units.
     💬 Tuesdays tend to be on the quieter side for this category.

  📅 Wednesday, March 04
     🌡️ 51°F  🌧️ | 🎫 Celtics Home Game, Bruins Home Game
     📦 Predicted: ~106 units
     🟢 Expecting a quieter Wednesday — forecast is ~106 units.
     💬 Wednesdays tend to be on the quieter side for this category. Sales have been trending downward this week, so we're being a bit more conservative. Temperature is swinging 29°F warmer compared to yesterday — sudden changes like this often shift what people buy.

  📅 Thursday, March 05
     🌡️ 36°F  🌧️ | 🎫 Concert at House of Blues
     📦 Predicted: ~99 units
     🟢 Expecting a quieter Thursday — forecast is ~99 units.
     💬 Thursdays tend to be on the quieter side for this category.

  📅 Friday, March 06
     🌡️ 35°F  🌧️
     📦 Predicted: ~104 units
     🟢 Expectin

Generate a full WhatsApp message for tomorrow

In [8]:
# Generate the actual WhatsApp message for tomorrow
tomorrow = forecast.iloc[1]
tomorrow_date = pd.Timestamp(tomorrow["date"])

future_row = pd.DataFrame({
    "date": [tomorrow_date],
    "sales": [0],
    "onpromotion": [0],
})

combined = pd.concat([
    recent_sales[["date", "sales", "onpromotion"]],
    future_row
]).reset_index(drop=True)

# Filter weather to only matching dates
combined_dates = pd.to_datetime(combined["date"])
weather_for_combined = weather_hist[weather_hist["date"].isin(combined_dates)].copy()

weather_row = pd.DataFrame({
    "date": [tomorrow_date],
    "temp_high": [tomorrow["temp_high"]],
    "temp_feels_like": [tomorrow["temp_feels_like"]],
    "precipitation_mm": [tomorrow["precipitation_mm"]],
    "is_precipitation": [tomorrow["is_precipitation"]],
    "humidity": [65],
})

weather_for_combined = pd.concat([weather_for_combined, weather_row]).drop_duplicates(subset="date")

future_features = build_features(combined, holidays, weather_df=weather_for_combined).dropna()
last_row = future_features.iloc[-1]
feature_row = last_row[feat_cols]

prediction = max(model.predict(feature_row.values.reshape(1, -1))[0], 0)
result = get_shap_explanation(model, feature_row, feat_cols, tomorrow_date, prediction)

fake_inventory = int(prediction * 0.85)

msg = format_whatsapp_message(
    store_name="Dorchester Bodega",
    category="Beverages",
    date=tomorrow_date,
    prediction=prediction,
    explanation_result=result,
    current_inventory=fake_inventory,
)

print("📱 TOMORROW'S WHATSAPP ALERT")
print("=" * 50)
print(msg)

📱 TOMORROW'S WHATSAPP ALERT
🔮 *Dorchester Bodega — Beverages*
📅 Wednesday, March 04

*Expecting a quieter Wednesday — forecast is ~106 units.*

Wednesdays tend to be on the quieter side for this category. Sales have been trending downward this week, so we're being a bit more conservative. Temperature is swinging 29°F warmer compared to yesterday — sudden changes like this often shift what people buy.

🟢 Confidence: High

⚠️ *Heads up:* You have 89 units on hand but we're forecasting 106. Consider ordering 17 more.
